## Assignment 1: Supervised Machine Learning

Welcome to the first assignment of CS 541! This assignment prepares you with some useful tools that are widely used in NLP. This assignment must be done individually.

After this assignment, you should be able to:  
1. Load a dataset from huggingface's dataset library, and do some exploratory analyses.  
2. Use scikit-learn to build and train a feature-based model.  
3. Use pytorch to build and train a feature-based model.  
4. Use Optuna to automatically search for hyperparameters.  

In CS541, any work generated by an AI shouldn't be included without declaration. If you include material generated by an AI, the level of AI use should be properly documented, and the actual tool should be noted (e.g., "I used Codex to proofread the codes and draft the analysis"). 

**AI use declaration**
1. I have not used any type of AI for comments, all the comments are written by myself expect where I have specifically mentioned.
2. I have used Chatgpt Luna 5.2 for proofreading and get a overiview/glance of assignment.
3. I used Claude to draft the Optuna objective and then tweaked it, ran and verified it myself.

### 1. Load the dataset (5')
First, we are going to load the datasets from huggingface's `datasets` library.
Do some exploratory analysis on the dataset.  
1.1 Print out one example in the dataset. Briefly comment on what it contains.  
1.2 For each of the train, validation, and test set, compute the following statistics: 
- The number of data samples with each class label.  
- The mean and std of the sentence lengths (in words) of each `question`.  

1.3 Vectorize the validation set of the dataset, following the approaches specified in the train set example.

In [2]:
import pandas as pd 
import numpy as np 
from datasets import load_dataset  # huggingface datasets

ds = load_dataset("stanfordnlp/sst2") # loading the sst2 dataset from standfordnlp

c:\Users\manda\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [ ]:
# TODO -- Print out one example in the dataset. Briefly comment on what it contains.
from attr import has


example = ds["train"][2]
print(example)
print("\nEach example contains three fields: "
      "\n'idx' --> (the row index inside the split),"
      "\n'sentence' --> (movie review sentence or phrase in the dataset)"
      "\n'label' --> (0 means -ve sentiment, 1 means +ve sentiment).")

## Therefore, in the above example,
## has a row index of 2, 
## the sentence is "that loves its characters and communicates something rather beautiful about human nature"
## and the label is 1, which indicates that the sentiment of the sentence is positive.

{'idx': 2, 'sentence': 'that loves its characters and communicates something rather beautiful about human nature ', 'label': 1}

Each example contains three fields: 
'idx' --> (the row index inside the split),
'sentence' --> (movie review sentence or phrase in the dataset)
'label' --> (0 means -ve sentiment, 1 means +ve sentiment).


In [ ]:
# TODO -- Print out one example in the dataset. Briefly comment on what it contains.
from attr import has


example = ds["test"][3]
print(example)

## Therefore, in the above example,
## has a row index of 3, 
## the sentence is "director rob marshall went out gunning to make a great one ."
## and the label is -1, which indicates that there is no ground truth label available(hidden/unknown answer).

{'idx': 3, 'sentence': 'director rob marshall went out gunning to make a great one .', 'label': -1}


In [20]:
train_data = pd.DataFrame(ds["train"])
X_train_text = train_data["sentence"]
Y_train = train_data["label"]

val_data = pd.DataFrame(ds["validation"])
X_val_text = val_data["sentence"]
Y_val = val_data["label"]

test_data = pd.DataFrame(ds["test"])
X_test_text = test_data["sentence"]
Y_test = test_data["label"]

# This code separates the dataset into train, validation and test sets.
# Sentences represented by X (input)
# Labels represented by Y (output)

In [ ]:
# TODO -- compute the exploratory statistics
# This code splits the data in train, validation and test sets and calculates the 
# 1. Number of samples for each split
# 2. Class-label counts for each split
# 3. Mean sentence length (in words) for each split
# 4. Standard deviation of sentence length (in words) for each split


stats_rows = []  ## Creates empty list
for split_name in ["train", "validation", "test"]: # Goes thruogh all three dataset splits
    df = pd.DataFrame(ds[split_name])  ## converts the splits into dataframes
    sentence_lengths = df["sentence"].str.split().str.len()   # keeps the count of words in each sentence (not characters)
    # eg. My name is Mandar --> 4 words ["My", "name", "is", "Mandar"]

    class_counts = df["label"].value_counts().sort_index() # how many labels belong to 0 and 1

    print("\n===== {} ({} samples) =====".format(split_name, len(df))) # information of split and number of samples in it

    # print class counts 
    print("Class counts:")
    print(class_counts.to_string())

    # Average number of words per sentence upto 2 decimal places
    print("Mean sentence length (words): {:.4f}".format(sentence_lengths.mean()))
    # Standard deviation of number of words per sentence upto 2 decimal places
    print("Std  sentence length (words): {:.4f}".format(sentence_lengths.std()))

    # Stores the statistics in a dict and appnend to list of stats_rows
    stats_rows.append({"split": split_name, "n": len(df),
                       **{"label={}".format(k): v for k, v in class_counts.items()},
                       "mean_len_words": sentence_lengths.mean(), 
                       "std_len_words": sentence_lengths.std()
                      })

print("\nSummary table:")
print(pd.DataFrame(stats_rows).fillna(0).to_string(index=False)) ## create and print the summary table


# We can clearly see that all the test labels are -1 which indicates that we cannot use it
# Therefore, for model selection we will use the validation set


===== train (67349 samples) =====
Class counts:
label
0    29780
1    37569
Mean sentence length (words): 9.4096
Std  sentence length (words): 8.0738

===== validation (872 samples) =====
Class counts:
label
0    428
1    444
Mean sentence length (words): 19.5482
Std  sentence length (words): 8.7639

===== test (1821 samples) =====
Class counts:
label
-1    1821
Mean sentence length (words): 19.2339
Std  sentence length (words): 8.9224

Summary table:
     split     n  label=0  label=1  mean_len_words  std_len_words  label=-1
     train 67349  29780.0  37569.0        9.409553       8.073806       0.0
validation   872    428.0    444.0       19.548165       8.763900       0.0
      test  1821      0.0      0.0       19.233937       8.922386    1821.0


In [19]:
## Tabular Understanding of the statistics calculated above
stats_table = (pd.DataFrame(stats_rows).fillna(0)
               .rename(columns={"n": "samples", "mean_len_words": "mean length (words)", "std_len_words": "std length (words)"}))
stats_table.round(2)


,split,samples,label=0,label=1,mean length (words),std length (words),label=-1
0,train,67349,29780.0,37569.0,9.41,8.07,0.0
1,validation,872,428.0,444.0,19.55,8.76,0.0
2,test,1821,0.0,0.0,19.23,8.92,1821.0


- Train(67,349): +ve → 56%, −ve → 44% → mildly imbalanced.
- Validation(872): +ve → 51%, −ve → 49% → almost balanced.
- Test(1,821): labels → −1 → hidden/unavailable for evaluation.
- Sentence length: Train ≈ 9.4 words; Val/Test ≈ 19 words.

Next we are going to vectorize the texts using TfidfVectorizer, then compute the Tf-idf features.   

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer 
from sklearn.neural_network import MLPClassifier 

counter = CountVectorizer(min_df=10, max_df=20) # Create the word count vectorizer
counter.fit(X_train_text)
print("Vocabulary size:", len(counter.vocabulary_)) # prints vocabulary size
X_train_counts = counter.transform(X_train_text)
print(X_train_counts.shape) # prints the shape of the training data
count2tfidf = TfidfTransformer(use_idf=True).fit(X_train_counts)
X_train = count2tfidf.transform(X_train_counts).toarray()
print(X_train.shape) # prints the shape of the transformed training data

Vocabulary size: 3120
(67349, 3120)
(67349, 3120)


In [ ]:
# TODO - Use the counter to convert X_val_text to occurrence vectors
# Note: don't create a new CountVectorizer, as we want to compute the vocabulary only on the train set
X_val_counts = counter.transform(X_val_text) # validation text -> word counts

# TODO - use count2tfidf to transform the counts into Tfidf features
# Note: don't create a new TfidfTransformer
X_val = count2tfidf.transform(X_val_counts).toarray() # validation counts --> TF-IDF features

print("Validation count shape:", X_val_counts.shape)
print("Validation TF-IDF shape:", X_val.shape)

Validation count shape: (872, 3120)
Validation TF-IDF shape: (872, 3120)


### 2. Train scikit-learn models (10')
Train a two-layer MLPClassifier using `random_state=0`. Manually tune the hyperparameters on the validation set. Report the procedure of hyperparameter tuning. Specifically: report the hyperparameters you have tried, and their results.  

After you are satisfied with the validation set performances, report the validation set performance. Use this set of hyperparameters and repeat the model training procedure for five times using `random_state` as 1, 2, 3, 31, 42 respectively. Record the five accuracy numbers.

In [ ]:
# Starter
import time
import warnings
from sklearn.exceptions import ConvergenceWarning

# This convergence warning issue waas resolved chatgpt
# We deliberately train with a fixed epoch budget (max_iter) rather than running to convergence, so the
# ConvergenceWarning that sklearn raises at the end of every fit is expected; it is silenced to keep the log readable.
warnings.filterwarnings("ignore", category=ConvergenceWarning)


#   step 1 baseline | step 2 hidden size | step 3 learning rate | step 4 batch size | step 5 epoch budget | step 6 L2 penalty
SKLEARN_TUNING_CONFIGS = [
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},  # 1 baseline
    {"hidden_layer_sizes": (50,),  "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},  # 2 hidden size
    {"hidden_layer_sizes": (200,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.0003, "batch_size": 128, "max_iter": 15, "alpha": 1e-4},  # 3 learning rate
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.003,  "batch_size": 128, "max_iter": 15, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 32,  "max_iter": 15, "alpha": 1e-4},  # 4 batch size
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 256, "max_iter": 15, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 5,  "alpha": 1e-4},  # 5 epoch budget
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 30, "alpha": 1e-4},
    {"hidden_layer_sizes": (100,), "learning_rate_init": 0.001,  "batch_size": 128, "max_iter": 15, "alpha": 1e-2},  # 6 L2 penalty
]
SKLEARN_SEEDS = [1, 2, 3, 31, 42] # five random_state values required by professor

def train_sklearn_model(X_train, Y_train, X_val, Y_val): 
    global sklearn_tuning_df, sklearn_best_params, sklearn_seed_accuracies # store the result in variables

    # Step 1: Manual Tuning 
    print("Manual hyperparameter tuning (random_state=0)")
    rows = []
    for config in SKLEARN_TUNING_CONFIGS: # try every configuration
        t0 = time.time() # measures training time
        model = MLPClassifier(random_state=0, **config) # create MLP model
        model.fit(X_train, Y_train) # Train the model using the training features and labels
        acc = model.score(X_val, Y_val) # Evaluate on validation data
        rows.append({**config, "validation_accuracy": acc, "seconds": round(time.time() - t0, 1)})
        print(config, "-> val acc = {:.4f}".format(acc))
    # Creates the tuning results table
    sklearn_tuning_df = pd.DataFrame(rows) 
    print("\nTuning results (all random_state=0):")
    print(sklearn_tuning_df.to_string(index=False))



    # Step 2: Select the best config (based highest validation accuracy)
    best = sklearn_tuning_df.loc[sklearn_tuning_df["validation_accuracy"].idxmax()] # finds row with highest validation accuracy

    # Extract the best performing rows and store them in dict
    sklearn_best_params = {
        "hidden_layer_sizes": tuple(best["hidden_layer_sizes"]),
        "learning_rate_init": float(best["learning_rate_init"]),
        "batch_size": int(best["batch_size"]),
        "max_iter": int(best["max_iter"]),
        "alpha": float(best["alpha"]),
    }
    print("\nSelected hyperparameters:", sklearn_best_params)

    # Train a new MLP using selected best hyperparameters with random_state=0
    final_model = MLPClassifier(random_state=0, **sklearn_best_params)
    final_model.fit(X_train, Y_train)
    final_val_accuracy = final_model.score(X_val, Y_val)
    print("Final validation accuracy with selected hyperparameters (random_state=0): {:.4f}".format(final_val_accuracy))



    # Step 3: repeat with the five required random states and same hyperparameters
    sklearn_seed_accuracies = []  # empty list for storing five validation accuracies
    print("\nRepeating with random_state in", SKLEARN_SEEDS)
    for seed in SKLEARN_SEEDS: # feeding the 5 given seeds and training it
        model = MLPClassifier(random_state=seed, **sklearn_best_params)
        model.fit(X_train, Y_train)
        acc = model.score(X_val, Y_val)
        sklearn_seed_accuracies.append(acc)
        print("random_state = {:>2d}: validation accuracy = {:.4f}".format(seed, acc))
    print("Five-seed mean = {:.4f}, std = {:.4f}".format(np.mean(sklearn_seed_accuracies),
                                                        np.std(sklearn_seed_accuracies, ddof=1)))
    return final_val_accuracy

train_sklearn_model(X_train, Y_train, X_val, Y_val)

Manual hyperparameter tuning (random_state=0)
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5780
{'hidden_layer_sizes': (50,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5700
{'hidden_layer_sizes': (200,), 'learning_rate_init': 0.001, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5803
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.0003, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5745
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.003, 'batch_size': 128, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5803
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001, 'batch_size': 32, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5826
{'hidden_layer_sizes': (100,), 'learning_rate_init': 0.001, 'batch_size': 256, 'max_iter': 15, 'alpha': 0.0001} -> val acc = 0.5745
{'hidden_layer_sizes': (100,), 

0.5825688073394495

**Q2 Report:- Tuning procedure and the obtained results.**

*Model* `MLPClassifier(hidden_layer_sizes=(h,))` is a two-layer fully-connected network (input → hidden → output) trained with the default Adam solver; `random_state=0` for every tuning run.

*Procedure.* 
- *Baseline*: Start with 100 neurons, learning rate 0.001, batch size 128, 15 epochs, alpha 0.0001.
- *Tuning*: Changes one hyperparameter at a time and checks validation accuracy 
- *Selection*: Chooses the combination that gives the highest validation accuracy.
- *Five seeds*: Keeps those selected settings fixed and trains the model 5 times with seeds 1, 2, 3, 31, 42 to see how much the result changes due to randomness


### 3. Train a pytorch model (10')
Here you will repeat the training of a two-layer fully-connected neural network using pytorch. Following are some specifications that may be helpful:  
- For each of the train and validation set, specify a dataloader, preferrably using `torch.utils.data.DataLoader`.  
- Use an optimizer of your choice. Adam, AdamW and SGD are popular choices.  
- Designate a number, `train_epochs`, as the number of passes through the dataset during training. Each pass through the training dataset is called an epoch.  
  - During the epoch, there may be many steps. In each step, load a batch of data from the dataloader. Compute the loss. Do a `backward()` pass to compute the gradients. Call a `step()` from the optimizer to update the model's parameters. Then zero out the gradients.
- At the end of each epoch, go through a validation run. Do *not* optimize the model during the validation run. Compute the accuracy of the model on this validation run, and print it out.

Tune the hyperparameters on the validation set. Report the hyperparameters you have tried, and their results. 

After you are satisfied with the validation set performances, record the set of hyperparameters. Use this set of hyperparameters, and repeat the model training procedure for five times using 1, 2, 3, 31, 42 as random seeds respectively. You can use `torch.manual_seed()` to set the random seeds. Record the five accuracy numbers.

In [ ]:
# Starter
# Import necessary PyTorch tools and modules
import torch
import torch.nn as nn
from collections import OrderedDict

# Define fully-connected NN where the list all_layer_sizes determines, input -> hidden -> output layers
class MLP(nn.Module):
    def __init__(self, all_layer_sizes):
        super().__init__()
        
        # Create the layers 
        layers = OrderedDict()
        for i in range(len(all_layer_sizes) - 1):
            # Create the Linear layer that connect current layer to the next layer
            layers["linear_{}".format(i)] = nn.Linear(all_layer_sizes[i], all_layer_sizes[i + 1]) # Performs NN calculations
            if i < len(all_layer_sizes) - 2:
                layers["relu_{}".format(i)] = nn.ReLU() # Relu acts as activation function between layers
        self.net = nn.Sequential(layers)

    def forward(self, X):
        return self.net(X)

# Takes batch of examples and return tuple of two tensors
def my_collate_function(batch):
    batch_X, batch_Y = [], []
    for item in batch:
        batch_X.append(item[0])
        batch_Y.append(item[1])
    # Stack into one numpy array first: building a tensor from a list of numpy arrays is very slow (PyTorch warns about it).
    return torch.tensor(np.array(batch_X)).float(), torch.tensor(np.array(batch_Y)).long()


# Pairs input with their labels
def prepare_zipped_XY(X, Y):
    zipped = []
    for i in range(len(X)):
        zipped.append((X[i], Y[i]))
    return zipped


# Zip the (dense numpy) features with the labels once and and reuse whenever needed
train_zipped = prepare_zipped_XY(X_train, np.asarray(Y_train))
val_zipped = prepare_zipped_XY(X_val, np.asarray(Y_val))
N_CLASSES = 2

#Set PyTorch hyperparams
PYTORCH_HPARAMS = {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001, "hidden_sizes": [100]}
PYTORCH_SEED = 1


def train_pytorch_model(X_train, Y_train, X_val, Y_val):
    # Define the manual seed
    torch.manual_seed(PYTORCH_SEED)

    # Read the hyperparameters
    train_epochs = PYTORCH_HPARAMS["train_epochs"]
    batch_size = PYTORCH_HPARAMS["batch_size"]
    learning_rate = PYTORCH_HPARAMS["learning_rate"]
    hidden_sizes = list(PYTORCH_HPARAMS["hidden_sizes"])

    # Set up the model, optimizer and dataloader
    model = MLP([X_train.shape[1]] + hidden_sizes + [N_CLASSES])
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    train_dataloader = torch.utils.data.DataLoader(train_zipped, batch_size=batch_size, shuffle=True,
                                                   collate_fn=my_collate_function)
    val_dataloader = torch.utils.data.DataLoader(val_zipped, batch_size=batch_size, shuffle=False,
                                                 collate_fn=my_collate_function)
    print("Start training!  seed={}  hparams={}".format(PYTORCH_SEED, PYTORCH_HPARAMS))
    last_epoch_dev_acc = 0
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            logits = model(batch_X)                 # forward pass
            loss = loss_function(logits, batch_Y)   # compute the loss
            loss.backward()                         # backward pass -> gradients
            optim.step()                            # update the parameters
            optim.zero_grad()                       # zero out the gradients

        # End-of-epoch validation run: eval mode, no gradients, and NO optimizer calls.
        n_correct, n_total = 0, 0
        model.eval()
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                predictions = torch.argmax(model(batch_X), dim=1)
                n_correct += (predictions == batch_Y).sum().item()
                n_total += batch_Y.size(0)

        last_epoch_dev_acc = n_correct/n_total
        print("Epoch {}, val accuracy {:.4f}".format(epoch+1, last_epoch_dev_acc))

    return last_epoch_dev_acc

train_pytorch_model(X_train, Y_train, X_val, Y_val)

Start training!  seed=1  hparams={'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.001, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5826
Epoch 2, val accuracy 0.5688
Epoch 3, val accuracy 0.5803
Epoch 4, val accuracy 0.5780
Epoch 5, val accuracy 0.5791
Epoch 6, val accuracy 0.5803
Epoch 7, val accuracy 0.5745
Epoch 8, val accuracy 0.5757


0.5756880733944955

### PyTorch Model
- Defines a **two-layer fully-connected neural network** with 100 hidden neurons, ReLU activation and 2 output classes
- Uses **Adam optimizer and CrossEntropyLoss** to train the model batch-by-batch for 8 epochs.
- After each epoch, the model is evaluated on the **validation set without updating the model** and the validation accuracy is printed

In [ ]:
# PyTorch tuning configurations
PYTORCH_TUNING_CONFIGS = [ 
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [100]},  # 1 baseline
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.0003, "hidden_sizes": [100]},  # 2 learning rate
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.003,  "hidden_sizes": [100]},
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [50]},   # 3 hidden size
    {"train_epochs": 8, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [200]},
    {"train_epochs": 8, "batch_size": 64,  "learning_rate": 0.001,  "hidden_sizes": [100]},  # 4 batch size
    {"train_epochs": 4, "batch_size": 128, "learning_rate": 0.001,  "hidden_sizes": [100]},  # 5 epoch budget
]

pytorch_tuning_rows = []
for config in PYTORCH_TUNING_CONFIGS:
    PYTORCH_HPARAMS = config
    PYTORCH_SEED = 1
    acc = train_pytorch_model(X_train, Y_train, X_val, Y_val)
    
    #Store each result
    pytorch_tuning_rows.append({**config, "hidden_sizes": str(config["hidden_sizes"]), "validation_accuracy": acc})
    print("-> config {} : last-epoch val acc = {:.4f}\n".format(config, acc))

# Create the tuning-results table
pytorch_tuning_df = pd.DataFrame(pytorch_tuning_rows)
print("PyTorch tuning results (seed 1):")
print(pytorch_tuning_df.to_string(index=False))

# Select the best configuration
best_row = pytorch_tuning_df.loc[pytorch_tuning_df["validation_accuracy"].idxmax()]

# Save the selected hyperparameters
pytorch_best_params = {
    "train_epochs": int(best_row["train_epochs"]),
    "batch_size": int(best_row["batch_size"]),
    "learning_rate": float(best_row["learning_rate"]),
    "hidden_sizes": eval(best_row["hidden_sizes"]),
}
print("\nSelected PyTorch hyperparameters:", pytorch_best_params)

Start training!  seed=1  hparams={'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.001, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5826
Epoch 2, val accuracy 0.5688
Epoch 3, val accuracy 0.5803
Epoch 4, val accuracy 0.5780
Epoch 5, val accuracy 0.5791
Epoch 6, val accuracy 0.5803
Epoch 7, val accuracy 0.5745
Epoch 8, val accuracy 0.5757
-> config {'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.001, 'hidden_sizes': [100]} : last-epoch val acc = 0.5757

Start training!  seed=1  hparams={'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.0003, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5791
Epoch 2, val accuracy 0.5849
Epoch 3, val accuracy 0.5803
Epoch 4, val accuracy 0.5757
Epoch 5, val accuracy 0.5745
Epoch 6, val accuracy 0.5803
Epoch 7, val accuracy 0.5780
Epoch 8, val accuracy 0.5814
-> config {'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.0003, 'hidden_sizes': [100]} : last-epoch val acc = 0.5814

Start training!  seed=1  hparams={'train_e

### PyTorch Hyperparameter Tuning Report
- Tests different **learning rates, hidden-layer sizes, batch sizes and number of epochs**, starting from a baseline configuration
- Each configuration is trained with the **same random seed (1)** and compared using validation accuracy
- The configuration with the **highest validation accuracy** is selected as the final set of PyTorch hyperparameters

In [11]:
PYTORCH_SEEDS = [1, 2, 3, 31, 42]
PYTORCH_HPARAMS = pytorch_best_params

pytorch_seed_accuracies = []
for seed in PYTORCH_SEEDS:
    PYTORCH_SEED = seed
    acc = train_pytorch_model(X_train, Y_train, X_val, Y_val)
    pytorch_seed_accuracies.append(acc)

print("\nSelected hyperparameters:", pytorch_best_params)
for seed, acc in zip(PYTORCH_SEEDS, pytorch_seed_accuracies):
    print("torch.manual_seed({:>2d}): validation accuracy = {:.4f}".format(seed, acc))
print("Five-seed mean = {:.4f}, std = {:.4f}".format(np.mean(pytorch_seed_accuracies),
                                                    np.std(pytorch_seed_accuracies, ddof=1)))

Start training!  seed=1  hparams={'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.003, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5849
Epoch 2, val accuracy 0.5745
Epoch 3, val accuracy 0.5837
Epoch 4, val accuracy 0.5803
Epoch 5, val accuracy 0.5837
Epoch 6, val accuracy 0.5814
Epoch 7, val accuracy 0.5849
Epoch 8, val accuracy 0.5826
Start training!  seed=2  hparams={'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.003, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5837
Epoch 2, val accuracy 0.5803
Epoch 3, val accuracy 0.5849
Epoch 4, val accuracy 0.5791
Epoch 5, val accuracy 0.5860
Epoch 6, val accuracy 0.5872
Epoch 7, val accuracy 0.5883
Epoch 8, val accuracy 0.5860
Start training!  seed=3  hparams={'train_epochs': 8, 'batch_size': 128, 'learning_rate': 0.003, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5780
Epoch 2, val accuracy 0.5860
Epoch 3, val accuracy 0.5791
Epoch 4, val accuracy 0.5780
Epoch 5, val accuracy 0.5780
Epoch 6, val accuracy 0.5768
Epoch

### PyTorch Five-Seed Evaluation Report
- Uses the **selected best hyperparameters** and trains the model five times with random seeds **1, 2, 3, 31, and 42**.
- Records the validation accuracy from each run to see how much the model performance changes with different random seeds.
- Finally, calculates the **mean and standard deviation** of the five validation accuracies.

### 4. Hyperparameter tuning (10')
This question requires modifying your previous pytorch training scripts. Use Optuna to find the hyperparameters that can maximize the accuracy on the validation set.  

The range of hyperparameters don't need to be too large (i.e., the total program should still be runnable within a reasonable time). The most important hyperparameter is the learning rate. Other hyperparameters that you can tune include the train epochs, batch size, hidden sizes, etc.  

When you are satisfied with the hyperparameters, report the hyperparameter and the resulting validation accuracy.

In [ ]:
# Starter
import optuna 

# Optuna chooses the hyperparameters for each trial
def train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val):
    # TODO -- Modify your train_pytorch_model() in the previous section, so that some hyperparameters are recommended from the Optuna

    
    # Define the hyperparameter search space
    train_epochs = trial.suggest_int("train_epochs", 4, 10)
    batch_size = trial.suggest_categorical("batch_size", [64, 128, 256])
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    hidden_size = trial.suggest_categorical("hidden_size", [50, 100, 200])


    # Keeping the seed fixed so trials are compared consistently
    torch.manual_seed(1)   
    
    model = MLP([X_train.shape[1], hidden_size, N_CLASSES])
    optim = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    
    # created training and validation batches
    train_dataloader = torch.utils.data.DataLoader(train_zipped, batch_size=batch_size, shuffle=True, collate_fn=my_collate_function)
    val_dataloader = torch.utils.data.DataLoader(val_zipped, batch_size=batch_size, shuffle=False, collate_fn=my_collate_function)


    val_acc = 0.0
    
    # Train and evalute the model for each epoch
    for epoch in range(train_epochs):
        model.train()
        for batch_X, batch_Y in train_dataloader:
            logits = model(batch_X)
            loss = loss_function(logits, batch_Y)
            loss.backward()
            optim.step()
            optim.zero_grad()
        model.eval()
        n_correct, n_total = 0, 0
        
        # Evaluating of validation data without any updation in model
        with torch.no_grad():
            for batch_X, batch_Y in val_dataloader:
                predictions = torch.argmax(model(batch_X), dim=1)
                n_correct += (predictions == batch_Y).sum().item()
                n_total += batch_Y.size(0)
        val_acc = n_correct / n_total
        trial.report(val_acc, epoch)           # lets Optuna prune clearly hopeless trials early
        if trial.should_prune():
            raise optuna.TrialPruned()
    return val_acc

def find_optimal_hyper_params(X_train, Y_train, X_val, Y_val):
    # Start an Optuna study
    # direction="maximize" is essential as Optuna minimises by default and our main goal is accuracy
    study = optuna.create_study(direction="maximize",
                                sampler=optuna.samplers.TPESampler(seed=0),
                                pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))

    # TODO -- objective is a function that takes only one argument. Need to also pass in the other arguments. 
    # Hint: You can define another function within the scope of this function
    def objective(trial):
        return train_pytorch_model_with_optuna(trial, X_train, Y_train, X_val, Y_val)
    study.optimize(objective, n_trials=20) # test 20 different hyperparameter configs

    # Best configs will only be displayed
    print("\nAll trials:")
    print(study.trials_dataframe(attrs=("number", "value", "params", "state")).to_string(index=False))
    print("\nBest hyperparameters:", study.best_params)
    print("Best validation accuracy: {:.4f}".format(study.best_value))
    return study

optuna_study = find_optimal_hyper_params(X_train, Y_train, X_val, Y_val)

[I 2026-09-16 21:54:50,890] A new study created in memory with name: no-name-552a6126-f288-4539-9e63-38b4b3e9e993
[I 2026-09-16 21:55:11,334] Trial 0 finished with value: 0.5837155963302753 and parameters: {'train_epochs': 7, 'batch_size': 64, 'learning_rate': 0.0007035737028722148, 'hidden_size': 200}. Best is trial 0 with value: 0.5837155963302753.
[I 2026-09-16 21:55:31,246] Trial 1 finished with value: 0.5825688073394495 and parameters: {'train_epochs': 10, 'batch_size': 128, 'learning_rate': 0.001368009527972693, 'hidden_size': 50}. Best is trial 0 with value: 0.5837155963302753.
[I 2026-09-16 21:55:36,583] Trial 2 finished with value: 0.5825688073394495 and parameters: {'train_epochs': 4, 'batch_size': 256, 'learning_rate': 0.009062263471261952, 'hidden_size': 50}. Best is trial 0 with value: 0.5837155963302753.
[I 2026-09-16 21:55:42,563] Trial 3 finished with value: 0.5802752293577982 and parameters: {'train_epochs': 4, 'batch_size': 256, 'learning_rate': 0.001105851072569646, 


All trials:
 number    value  params_batch_size  params_hidden_size  params_learning_rate  params_train_epochs    state
      0 0.583716                 64                 200              0.000704                    7 COMPLETE
      1 0.582569                128                  50              0.001368                   10 COMPLETE
      2 0.582569                256                  50              0.009062                    4 COMPLETE
      3 0.580275                256                 200              0.001106                    4 COMPLETE
      4 0.576835                256                 100              0.001676                    7 COMPLETE
      5 0.576835                128                  50              0.002155                    6   PRUNED
      6 0.577982                128                 100              0.009479                    6 COMPLETE
      7 0.576835                128                 100              0.000208                    8 COMPLETE
      8 0.58600

# Optuna Results — Observations 
#### Did analysis on own, used ai just to look and make it more visually appealing

- **Best validation accuracy:** ≈ 58.60% → obtained with **5 epochs, batch size 128, learning rate ≈ 0.00474, hidden size 100**.
- **Compared with baseline:** Baseline ≈ 58.37% → Optuna improves accuracy by only ≈ **0.23 percentage points**.
- **Learning rate:** Best trial uses ≈ 0.0047, but other good results occur at different learning rates → no clear single trend.
- **Hidden size:** 100 and 200 both appear among the better trials → no strong conclusion that larger/smaller is always better.
- **Batch size:** 64, 128 and 256 all appear in completed trials → performance varies rather than following a clear trend.
- **Pruning:** Some trials were **PRUNED** → Optuna stopped trials whose intermediate validation performance was not promising.
- **Overall:** Optuna found a configuration around **58.6% validation accuracy**, but the improvement over the baseline is small.

### 5. Bonus: Compare the performances of the two methods (2')
Use an appropriate $t$ test, compare the five performance numbers of the sklearn model and the pytorch model *under the same set of hyperparameters*. Do their results differ?

Note: The scores for bonus will be added to the A1 total score, but the total score will be capped to 100%.

In [ ]:
# Bonus: paired t-test between sklearn and PyTorch under the SAME hyperparameters
from scipy.stats import ttest_rel, ttest_ind

# Using the same hyperparameters for both models
shared_hparams = {
    "train_epochs": sklearn_best_params["max_iter"],
    "batch_size": sklearn_best_params["batch_size"],
    "learning_rate": sklearn_best_params["learning_rate_init"],
    "hidden_sizes": list(sklearn_best_params["hidden_layer_sizes"]),
}
print("Shared hyperparameters:", shared_hparams)

# Run PyTorch with the same five seeds used for sklearn
PYTORCH_HPARAMS = shared_hparams
pytorch_same_hp_accuracies = []

for seed in SKLEARN_SEEDS:         
    PYTORCH_SEED = seed
    pytorch_same_hp_accuracies.append(train_pytorch_model(X_train, Y_train, X_val, Y_val))


# Compare the five validation accuracies and their mean
print("\nsklearn accuracies (random_state 1,2,3,31,42):", np.round(sklearn_seed_accuracies, 4))
print("PyTorch accuracies (manual_seed  1,2,3,31,42):", np.round(pytorch_same_hp_accuracies, 4))
print("sklearn mean = {:.4f}, PyTorch mean = {:.4f}".format(np.mean(sklearn_seed_accuracies), np.mean(pytorch_same_hp_accuracies)))


# Paired t-test compares the results seed-by-seed
t_stat, p_value = ttest_rel(sklearn_seed_accuracies, pytorch_same_hp_accuracies)
print("\nPaired t-test:  t = {:.4f}, p = {:.4f}".format(t_stat, p_value))


# Welch's test provide independent samples comparison
t_ind, p_ind = ttest_ind(sklearn_seed_accuracies, pytorch_same_hp_accuracies, equal_var=False)
print("Welch's t-test: t = {:.4f}, p = {:.4f}".format(t_ind, p_ind))

alpha = 0.05
if p_value < alpha:
    print("\nAt alpha = 0.05 the difference between the two implementations IS statistically significant.")
else:
    print("\nAt alpha = 0.05 there is NOT enough evidence that the two implementations differ.")

Shared hyperparameters: {'train_epochs': 15, 'batch_size': 32, 'learning_rate': 0.001, 'hidden_sizes': [100]}
Start training!  seed=1  hparams={'train_epochs': 15, 'batch_size': 32, 'learning_rate': 0.001, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5826
Epoch 2, val accuracy 0.5677
Epoch 3, val accuracy 0.5814
Epoch 4, val accuracy 0.5803
Epoch 5, val accuracy 0.5814
Epoch 6, val accuracy 0.5814
Epoch 7, val accuracy 0.5803
Epoch 8, val accuracy 0.5849
Epoch 9, val accuracy 0.5814
Epoch 10, val accuracy 0.5803
Epoch 11, val accuracy 0.5826
Epoch 12, val accuracy 0.5837
Epoch 13, val accuracy 0.5860
Epoch 14, val accuracy 0.5860
Epoch 15, val accuracy 0.5872
Start training!  seed=2  hparams={'train_epochs': 15, 'batch_size': 32, 'learning_rate': 0.001, 'hidden_sizes': [100]}
Epoch 1, val accuracy 0.5837
Epoch 2, val accuracy 0.5791
Epoch 3, val accuracy 0.5791
Epoch 4, val accuracy 0.5768
Epoch 5, val accuracy 0.5768
Epoch 6, val accuracy 0.5757
Epoch 7, val accuracy 0.5780
Epoch 8,

- Same hyperparameters + 5 seeds were used for sklearn and PyTorch.
- sklearn mean accuracy ≈ 58.28%; PyTorch ≈ 58.33%.
- Paired t-test: p = 0.8466 > 0.05 → no significant performance difference between sklearn and PyTorch.

# CONCLUSION


*I explained gpt whatever I understood and took help from it to format my conclusion accordingly

- The SST-2 data consists of 67,349 training samples, 872 validation samples and 1,821 test samples. The training data set is slightly imbalanced, while the validation data set is nearly balanced. The test labels are -1, therefore, the test set cannot be used for evaluation.

- The text has been transformed to numbers through CountVectorizer and TF-IDF. There are 3,120 features in the vocabulary and CountVectorizer and TF-IDF transformations have been applied to the validation data set properly.

- For the sklearn model, the different values of hidden sizes, learning rates, batch sizes, epoch budgets and L2 penalties have been tried. The maximum validation accuracy is approximately 58.26%, with the batch size of 32, learning rate 0.001, hidden size 100 and 15 epochs. For the five different random seeds, the mean accuracy is approximately 58.28%.

- For the PyTorch model, the same two-layer neural network has been implemented using DataLoaders, Adam optimizer and CrossEntropyLoss. The hyperparameters have been tuned manually, and the optimal configuration resulted in the validation accuracy of approximately 58.26%. For the five different random seeds, the mean accuracy is approximately 58.17%.

- Then the Optuna has been used for automatic tuning of the PyTorch hyperparameters.